In [8]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

### Download an example reference data point from LangSmith

In [9]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [10]:
ls_client = Client()

In [11]:
dataset = ls_client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [12]:
dataset

Dataset(name='rag-evaluation-dataset', description='RAG evaluation dataset', data_type=<DataType.kv: 'kv'>, id=UUID('2bd631e3-f198-48d3-9b81-fb5c113074b9'), created_at=datetime.datetime(2026, 6, 28, 18, 53, 39, 86367, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 6, 28, 18, 53, 39, 86367, tzinfo=TzInfo(0)), example_count=31, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-15.7.5-x86_64-i386-64bit', 'sdk_version': '0.9.3', 'runtime_version': '3.12.13', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [13]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

{'ground_truth': 'Among these options, the Wekily earbuds and the Jesebang earbuds both offer up to 40 hours total playtime with their charging cases. The pamu earbuds offer up to 30 hours total battery life. If battery life is your main priority, the Wekily and Jesebang models are the strongest choices here.',
 'reference_context_ids': ['B0BRV544MV', 'B09X9838WY', 'B09TFM1SFQ'],
 'reference_descriptions': ['Wekily Bluetooth 5.3 Headphones, Wireless Earbuds with 40H Playtimes Charge Case, Deep Bass, IPX7 Waterproof Running Earphones with 4-Mic Clear Call, LED Display, in Ear Headphones for Work/Gym Excellent Sound Quality: Wireless earbuds adopte graphene drivers and a polymer composite diaphragm carrying an AAC/SBC audio decoder, resulting in up to 43% bass enhancement. Audio technology is losslessly transmitted for a clearer stereo sound from the headphones. 40Hrs Cycle Playtimes: The 400mAh charging case has a playtime of 35 hours and the wireless headphones have a single playtime o

In [14]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs

{'question': 'Which wireless earbuds you have offer the longest battery life?'}

In [15]:
reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs
reference_output = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

### RAG Pipeline

In [16]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    return response.choices[0].message.content


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [17]:
rag_pipeline("Can I get a charger?")

{'answer': 'Yes. We have:\n\n1) iPhone Charger 6ft 3Pack (Apple MFi Certified Lightning Cables) – fast charging up to 3A with high-speed data sync, reinforced durability, and compatibility with many iPhone and iPad models.\n\n2) USB-C to USB-C Cable (INIU) – 6.6ft, 100W PD 5A fast charging for USB-C devices like phones, tablets, and many laptops.\n\n3) Replacement Notebook Charger (White-60W) – compatible with notebooks; it’s the second generation of the power adapter (double-check your Mac notebook model before buying).\n\nWhich device are you charging (iPhone Lightning, USB-C laptop/phone, or a notebook)?',
 'question': 'Can I get a charger?',
 'retrieved_context_ids': ['B0BBVJJRHD',
  'B0BGH3H1WM',
  'B0BN1CMWCP',
  'B0C9QZS95R',
  'B0BXC72RLD'],
 'retrieved_context': ['iPhone Charger 6ft 3Pack, Apple MFi Certified Lightning Cable Fast Charging Long iPhone Charger Cord High Speed Data Sync Cable Compatible iPhone 14 13 12 11 Pro Max XS XR X 8 7 6S 6 Plus SE 5S, iPad 【iPhone Cable Fa

### RAGAS Metrics

In [18]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_85215/3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_85215/3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_85215/3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is depre

In [19]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_85215/840510326.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_85215/840510326.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [ ]:
reference_input

In [20]:
reference_output

{'ground_truth': 'Among these options, the Wekily earbuds and the Jesebang earbuds both offer up to 40 hours total playtime with their charging cases. The pamu earbuds offer up to 30 hours total battery life. If battery life is your main priority, the Wekily and Jesebang models are the strongest choices here.',
 'reference_context_ids': ['B0BRV544MV', 'B09X9838WY', 'B09TFM1SFQ'],
 'reference_descriptions': ['Wekily Bluetooth 5.3 Headphones, Wireless Earbuds with 40H Playtimes Charge Case, Deep Bass, IPX7 Waterproof Running Earphones with 4-Mic Clear Call, LED Display, in Ear Headphones for Work/Gym Excellent Sound Quality: Wireless earbuds adopte graphene drivers and a polymer composite diaphragm carrying an AAC/SBC audio decoder, resulting in up to 43% bass enhancement. Audio technology is losslessly transmitted for a clearer stereo sound from the headphones. 40Hrs Cycle Playtimes: The 400mAh charging case has a playtime of 35 hours and the wireless headphones have a single playtime o

In [21]:
result = rag_pipeline(reference_input["question"])

In [22]:
result

{'answer': 'Among the wireless earbuds available, the longest total battery life is the Jesebang Wireless Earbud (B09X9838WY), with up to 40 hours of playtime (8 hours per single charge plus the charging case).',
 'question': 'Which wireless earbuds you have offer the longest battery life?',
 'retrieved_context_ids': ['B0BRV544MV',
  'B09X9838WY',
  'B0CFHWF326',
  'B09TFM1SFQ',
  'B0CH8DRD6K'],
 'retrieved_context': ['Wekily Bluetooth 5.3 Headphones, Wireless Earbuds with 40H Playtimes Charge Case, Deep Bass, IPX7 Waterproof Running Earphones with 4-Mic Clear Call, LED Display, in Ear Headphones for Work/Gym Excellent Sound Quality: Wireless earbuds adopte graphene drivers and a polymer composite diaphragm carrying an AAC/SBC audio decoder, resulting in up to 43% bass enhancement. Audio technology is losslessly transmitted for a clearer stereo sound from the headphones. 40Hrs Cycle Playtimes: The 400mAh charging case has a playtime of 35 hours and the wireless headphones have a single

In [23]:
async def ragas_context_precision_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [24]:
await ragas_context_precision_id_based(result, reference_output)

0.6

In [25]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [26]:
await ragas_context_recall_id_based(result, reference_output)

1.0

In [27]:
async def ragas_faithfulness(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_context"]
        )

    scorer = Faithfulness(llm=ragas_llm)
    
    return await scorer.single_turn_ascore(sample)

In [28]:
await ragas_faithfulness(result, reference_output)

0.6666666666666666

In [29]:
async def ragas_relevancy(run, example):

    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [30]:
await ragas_relevancy(result, reference_output)

np.float64(0.9321365425987805)